# Minimal repro: Unsloth DPO + Gemma 4 vision — RESOLVED Apr 30, 2026

**Status:** Resolved. End-to-end vision DPO training works on official Unsloth mainline post-merge of [PR #5199](https://github.com/unslothai/unsloth/pull/5199). See "Resolution" section at the bottom for the final working stack.

**Original symptom (Apr 26):** `DPOTrainer.train()` hung silently at `Tokenizing 0/52 [00:00<?, ? examples/s]` with both `dataset_num_proc=1` and `dataset_num_proc=2`. CPU at ~0%, GPU at 0%, no error, no progress.

**Setup:** Gemma 4 E4B 4-bit, vision layers unfrozen, LoRA r=16, fresh SFT-warmstart adapter (or fresh LoRA — either reproduces). Synthetic preference pairs: 1024x1024 RGB images + short prompts/chosen/rejected.

**Diagnostics that worked (function-level):**

- `processor(images=..., text=...)` direct call: returns in 0.1s
- `_UnslothDPOTrainer.process_row(features=pairs[0], ...)` direct call: returns in 0s
- Manual Python loop calling `process_row` over all pairs: returns in ~2s

**What hung originally:** `dataset.map(process_row)` inside `DPOTrainer._prepare_dataset` — the same function works fine outside `dataset.map`.

## Bug arc — three upstream + one usage-side

Three upstream bugs surfaced and resolved over 2 days via Unsloth issue [#5196](https://github.com/unslothai/unsloth/issues/5196) and contributor @datta0:

1. **Tokenization hang** (this notebook's original symptom) — fixed by `datta0/unsloth@dpo_mp_hang` branch
2. **Data collator schema mismatch** (`ValueError: ...input_ids`) — fixed in the same branch's second push
3. **AttributeError in vision tower forward** (`'bool' object has no attribute 'all'`) — was specific to `trl 0.22.2`; resolved by upgrading to `trl 0.24.0`

PR #5199 merged into `unslothai/unsloth` mainline on Apr 28.

A fourth issue surfaced when building real preference pairs on Apr 30 (post-merge):

4. **`ValueError: Image features and image tokens do not match, tokens: 0, features: 512`** at first training step. Cause: when constructing DPO preference pairs with raw text prompts (e.g., `prompt = "Describe the image"`), the prompt string contains no image-token placeholder, so `input_ids` has 0 image-token slots. The vision encoder produces N image features but has nowhere to inject them. SFT auto-handles this via `apply_chat_template`; DPO with raw string prompts does not. **Fix:** prepend `tokenizer.image_token` (Gemma 4: `<|image|>`, ID 258880) to the PROMPT string. Or construct prompts via `apply_chat_template` with conversation-style messages.

This is a usage-side requirement, not an Unsloth bug. Posted as follow-up comment on issue #5196 for future searchers.

## Lesson on commit pinning

Original "verified working" pin from Apr 27 was `datta0/unsloth@e96d05bad233052a30f894c5050eb7ec2e65ebc5`. After PR #5199 merged Apr 28, that commit was orphaned by force-push during PR cleanup — commit ID still resolves but is no longer reachable from any branch on the fork. Synthetic test still passed because the orphaned commit included enough fixes for that specific data shape.

Real DPO training on production data exposed that orphaned commit was MISSING fix #3's vision keys handling (added in `dc3e6a5`, which came after `6b11713` and before the merge).

Pinning to mainline post-merge is more stable than pinning to contributor branch tips:

- Branch tips can be force-pushed without warning
- Orphaned commits eventually get GC'd by GitHub
- Merge commits on official repos are immutable

**Final pin:** `unslothai/unsloth.git@4f9c8321a2136e62fd86fe722a544afd534334a5` (mainline post-merge of PR #5199).

## Environment

Modal A100 80GB, Linux, Python 3.12.

## 1. Install

In [ ]:
# Same install pattern as dpo-vision-repro, but with strict pins on top-level
# deps to defend against version drift since verification.
%uv pip install pillow==11.3.0 --system
%uv pip install transformers==5.5.0 --system
%uv pip install trl==0.24.0 --system
%uv pip install peft==0.19.1 --system
%uv pip install datasets==4.3.0 --system
%uv pip install unsloth-zoo==2026.4.9 --system

# Apply Datta's fix branch (--no-deps means won't touch torch)
%uv pip uninstall unsloth --system
%uv pip install "git+https://github.com/unslothai/unsloth.git" --no-deps --system

# Force torch+torchvision matching pair from cu129 index, last
%uv pip install --reinstall torch==2.11.0 torchvision==0.26.0 --index-url https://download.pytorch.org/whl/cu129 --system

!rm -rf /root/unsloth_compiled_cache ~/unsloth_compiled_cache


%uv pip install bitsandbytes==0.49.2 --system

Using Python 3.12.6 environment at: /usr/local
Resolved 1 package in 18ms
Uninstalled 1 package in 2ms
Installed 1 package in 60ms
 - pillow==12.1.1
 + pillow==11.3.0
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.6 environment at: /usr/local
Audited 1 package in 15ms
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.6 environment at: /usr/local
Resolved 76 packages in 105ms
Uninstalled 1 package in 1ms
Installed 1 package in 53ms
 - fsspec==2026.2.0
 + fsspec==2025.9.0
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.6 environment at: /usr/local
Audited 1 package in 21ms
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.6 environment at: /usr/local
Audited 1 package in 17ms
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.6 environment at: /usr/local
Resolved 88 packages in 111ms
Uninstalled 14 packages in 302ms


In [2]:
!grep -n "_DPO_VISION_KEYS\|_vision_keys\|dpo_trainer_fix_data_collator" /usr/local/lib/python3.12/site-packages/unsloth/models/rl_replacements.py

58:_DPO_VISION_KEYS = (
129:def dpo_trainer_fix_data_collator(call_args, extra_args):
148:RL_EXTRA_ARGS["dpo_trainer"].append(dpo_trainer_fix_data_collator)
224:    if all(_k in function for _k in _DPO_VISION_KEYS):
227:    _extra_columns = "".join(f'                "{_k}",\n' for _k in _DPO_VISION_KEYS)
248:    if all(_k in function for _k in _DPO_VISION_KEYS):
254:        for _k in _DPO_VISION_KEYS
273:    if all(_k in function for _k in _DPO_VISION_KEYS):
283:            *_DPO_VISION_KEYS,
310:def dpo_trainer_data_collator_vision_keys(call_args, extra_args):
314:    _vision_keys = str(_DPO_VISION_KEYS)
317:        "if not hasattr(DataCollatorForPreference, '_unsloth_vision_keys_patch'):\n"
327:        "        for _k in " + _vision_keys + ":\n"
346:        "    DataCollatorForPreference._unsloth_vision_keys_patch = True\n"
392:RL_EXTRA_ARGS["dpo_trainer"].append(dpo_trainer_data_collator_vision_keys)


In [3]:
import subprocess
result = subprocess.run(['pip', 'show', 'torch', 'torchvision'], capture_output=True, text=True)
print(result.stdout)

Name: torch
Version: 2.11.0+cu129
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License: BSD-3-Clause
Location: /usr/local/lib/python3.12/site-packages
Requires: cuda-bindings, cuda-toolkit, filelock, fsspec, jinja2, networkx, nvidia-cudnn-cu12, nvidia-cusparselt-cu12, nvidia-nccl-cu12, nvidia-nvshmem-cu12, setuptools, sympy, triton, typing-extensions
Required-by: accelerate, bitsandbytes, cut-cross-entropy, peft, torchaudio, torchvision, unsloth_zoo
---
Name: torchvision
Version: 0.26.0+cu129
Summary: image and video datasets and models for torch deep learning
Home-page: https://github.com/pytorch/vision
Author: PyTorch Core Team
Author-email: soumith@pytorch.org
License: BSD
Location: /usr/local/lib/python3.12/site-packages
Requires: numpy, pillow, torch
Required-by: 



In [4]:
%uv pip install bitsandbytes --system

Using Python 3.12.6 environment at: /usr/local
Audited 1 package in 14ms
Note: you may need to restart the kernel to use updated packages.


In [5]:
import subprocess
result = subprocess.run(["pip", "show", "unsloth", "trl"], capture_output=True, text=True)
print(result.stdout)


Name: unsloth
Version: 2026.4.8
Summary: 2-5X faster training, reinforcement learning & finetuning
Home-page: https://unsloth.ai
Author: Unsloth AI team
Author-email: info@unsloth.ai
License: 
Location: /usr/local/lib/python3.12/site-packages
Requires: nest-asyncio, pydantic, pyyaml, typer
Required-by: 
---
Name: trl
Version: 0.24.0
Summary: Train transformer language models with reinforcement learning.
Home-page: https://github.com/huggingface/trl
Author: 
Author-email: Leandro von Werra <leandro.vonwerra@gmail.com>
License: 
Location: /usr/local/lib/python3.12/site-packages
Requires: accelerate, datasets, transformers
Required-by: unsloth_zoo



In [6]:
import unsloth
import importlib.metadata as md
for pkg in ["bitsandbytes", "unsloth", "unsloth_zoo", "trl", "transformers", "datasets", "peft", "torch", "pillow"]:
    try:
        print(f"{pkg}: {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg}: NOT INSTALLED")

print(f"\nunsloth location: {unsloth.__file__}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
bitsandbytes: 0.49.2
unsloth: 2026.4.8
unsloth_zoo: 2026.4.9
trl: 0.24.0
transformers: 5.5.0
datasets: 4.3.0
peft: 0.19.1
torch: 2.11.0+cu129
pillow: 12.1.1

unsloth location: /usr/local/lib/python3.12/site-packages/unsloth/__init__.py


## 2. Imports

In [8]:
from unsloth import FastVisionModel       # MUST be first
from trl import DPOTrainer, DPOConfig
from datasets import Dataset
from PIL import Image
import torch
import time
import os

## 3. Load Gemma 4 E4B 4-bit + fresh LoRA adapter

Use Unsloth's `FastVisionModel.get_peft_model` (NOT `PeftModel.from_pretrained`, which fails on `Gemma4ClippableLinear` module resolution).

In [9]:
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/gemma-4-e4b-it",      
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)
print(f"Base loaded. GPU: {torch.cuda.memory_allocated()/1e9:.1f} GB")

model = FastVisionModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    target_modules="all-linear",
)
print("Trainable params:")
model.print_trainable_parameters()

==((====))==  Unsloth 2026.4.8: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100 80GB PCIe. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu129. CUDA: 8.0. CUDA Toolkit: 12.9. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

Base loaded. GPU: 10.9 GB
Trainable params:
trainable params: 41,222,144 || all params: 8,037,378,592 || trainable%: 0.5129


## 4. Synthetic preference pairs (52 examples)

Just blank 1024x1024 RGB PIL images + short text. Real images and real preferences are not needed to reproduce — the hang is structural, not data-dependent.

Schema: flat `{images: [PIL], prompt: str, chosen: str, rejected: str}` — this is what Unsloth's vision DPO `process_row` expects (it does NOT want chat-template content blocks).

In [20]:
PROMPT = f"{tokenizer.image_token}\nDescribe the image in one sentence."

def make_pair(idx: int):
    img = Image.new("RGB", (1024, 1024), color=(idx % 256, 128, 200))
    return {
        "images": [img],
        "prompt": PROMPT,
        "chosen": f"Image {idx} shows a solid color block.",
        "rejected": f"Image {idx} contains nothing useful at all.",
    }

pairs = [make_pair(i) for i in range(52)]
print(f"Built {len(pairs)} synthetic pairs")
print(f"Sample image: {pairs[0]['images'][0].size}")

Built 52 synthetic pairs
Sample image: (1024, 1024)


## 5. Diagnostic: confirm `process_row` works on its own

This call IS the function that `dataset.map` will invoke 52 times during DPOTrainer prep. It works fine in isolation.

In [21]:
from unsloth_compiled_cache.UnslothDPOTrainer import _UnslothDPOTrainer

start = time.time()
out = _UnslothDPOTrainer.process_row(
    features=pairs[0],
    processing_class=tokenizer,
    max_prompt_length=1024,
    max_completion_length=None,
    add_special_tokens=False,
)
print(f"Single call OK in {time.time()-start:.2f}s. Keys: {list(out.keys())}")

Single call OK in 0.03s. Keys: ['prompt_input_ids', 'pixel_values', 'chosen_input_ids', 'rejected_input_ids']


In [22]:
# Manual loop: call process_row on every pair. Should finish in ~2s.
start = time.time()
processed = []
for p in pairs:
    out = _UnslothDPOTrainer.process_row(
        features=p,
        processing_class=tokenizer,
        max_prompt_length=1024,
        max_completion_length=None,
        add_special_tokens=False,
    )
    processed.append(out)
print(f"Manual loop over {len(pairs)} pairs: {time.time()-start:.2f}s")

Manual loop over 52 pairs: 0.87s


In [23]:
# Inspect first sample's tokenized state
sample = pairs_ds[0]
print("Keys:", list(sample.keys()))
print()
print("Image type:", type(sample['images'][0]) if 'images' in sample else "no images key")
print()
print("Prompt:", sample.get('prompt', '<missing>')[:200])
print()
print("Chosen:", sample.get('chosen', '<missing>')[:200])
print()
print("Rejected:", sample.get('rejected', '<missing>')[:200])
print()

# After Trainer processing — check if image tokens get inserted
# Look at one of the pre-processed train batches
from torch.utils.data import DataLoader
dl = DataLoader(pairs_ds, batch_size=1, collate_fn=lambda x: x)
batch = next(iter(dl))
print("Batch keys:", list(batch[0].keys()) if isinstance(batch, list) else list(batch.keys()))

Keys: ['images', 'prompt', 'chosen', 'rejected']

Image type: <class 'PIL.PngImagePlugin.PngImageFile'>

Prompt: Describe the image in one sentence.

Chosen: Image 0 shows a solid color block.

Rejected: Image 0 contains nothing useful at all.

Batch keys: ['images', 'prompt', 'chosen', 'rejected']


In [ ]:
# Sanity check the tokenized dataset shape before training
import torch

# Inspect the raw dataset
print("=== Raw dataset (before DPOTrainer preprocessing) ===")
print(f"Length: {len(pairs_ds)}")
print(f"Columns: {pairs_ds.column_names}")
sample = pairs_ds[0]
print(f"\nSample keys: {list(sample.keys())}")
print(f"Image type: {type(sample['images'][0]).__name__}")
print(f"Prompt: {repr(sample['prompt'][:100])}")
print(f"  Contains image_token: {tokenizer.image_token in sample['prompt']}")
print(f"Chosen length: {len(sample['chosen'])} chars")
print(f"Rejected length: {len(sample['rejected'])} chars")

# Tokenize a sample manually to check what the processor produces
print("\n=== Manual tokenization check ===")
processed = tokenizer(
    text=sample['prompt'] + sample['chosen'],
    images=sample['images'],
    return_tensors="pt",
)
print(f"Output keys: {list(processed.keys())}")
print(f"input_ids shape: {processed['input_ids'].shape}")
print(f"pixel_values shape: {processed['pixel_values'].shape if 'pixel_values' in processed else 'NOT PRESENT'}")

# Count image tokens in the tokenized output
image_token_id = tokenizer.image_token_id
n_image_tokens = (processed['input_ids'] == image_token_id).sum().item()
print(f"Number of image_token slots in input_ids: {n_image_tokens}")

# Decode first ~200 tokens to see structure
print(f"\nFirst 50 tokens decoded:")
print(repr(tokenizer.tokenizer.decode(processed['input_ids'][0][:50].tolist())))

## 6. Resolved: DPOTrainer trains end-to-end (was: hangs at tokenization)

Wraps `pairs` in a `Dataset` and instantiates DPOTrainer. The trainer's `_prepare_dataset` calls `dataset.map(process_row, ...)` internally — that's where the original tokenization hang occurred.

**Original symptom (Apr 26):** Progress bar sat at `0/52 [00:00<?, ? examples/s]`, CPU dropped to ~0%, GPU at 0%, no progress, no error. Required kernel interrupt.

**Current behaviour (Apr 30, on the resolution stack):**
- Tokenization completes in ~10-15 seconds for 52 pairs
- Training proceeds through 39 steps
- Loss drops from `ln(2) ≈ 0.6931` baseline toward 0 (synthetic perturbations easy to distinguish)
- `rewards/accuracies` reaches 1.0 quickly
- `rewards/margins` climbs above 6.0 by mid-training

**Required for vision DPO to train (in addition to the upstream fixes from PR #5199):**

The PROMPT string must contain Gemma 4's image token. Without it, training fails at first step with `ValueError: Image features and image tokens do not match, tokens: 0, features: 512`. The DPO data pipeline doesn't auto-insert image tokens for raw string prompts the way SFT does.

```python
# Required form
PROMPT = f"{tokenizer.image_token}\nDescribe the image in one sentence."

# Wrong form (no image token, fails at training start)
# PROMPT = "Describe the image in one sentence."
```

The image token for Gemma 4 is `<|image|>` (ID 258880), accessible as `tokenizer.image_token`.

In [24]:
pairs_ds = Dataset.from_list(pairs)
print(f"Dataset: {len(pairs_ds)} examples, columns: {pairs_ds.column_names}")

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    processing_class=tokenizer,
    train_dataset=pairs_ds,
    data_collator=None,
    args=DPOConfig(
        beta=0.1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=5e-6,
        output_dir="/tmp/dpo-repro",
        max_length=2048,
        max_prompt_length=1024,
        remove_unused_columns=False,
        logging_steps=1,
        dataset_num_proc=1,    # also tried 2 — both hang
    ),
)

# fails here ↓
dpo_trainer.train()

Dataset: 52 examples, columns: ['images', 'prompt', 'chosen', 'rejected']


Extracting prompt in train dataset (num_proc=1):   0%|          | 0/52 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=1):   0%|          | 0/52 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/52 [00:00<?, ? examples/s]

[accelerate.utils.other|WARNING]Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 52 | Num Epochs = 3 | Total steps = 39
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 41,222,144 of 8,037,378,592 (0.51% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.
/usr/local/lib/python3.12/site-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/site-packages/bitsandbytes/_ops.py:18

Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected
1,0.693147,0.000000,0.000000,0.000000,0.000000,-140.827591,-189.921448,-8.229449,-7.303521
2,0.693147,0.000000,0.000000,0.000000,0.000000,-140.874939,-184.726212,-8.412458,-7.189271
3,0.647139,0.107900,-0.001102,0.500000,0.109002,-136.046738,-179.417175,-8.339906,-7.184594
4,0.721479,-0.009996,0.031765,0.250000,-0.041762,-147.729935,-194.353516,-7.775274,-6.924704
5,0.660545,0.171642,0.085666,0.500000,0.085976,-138.247055,-187.577621,-8.381559,-7.400293
6,0.671797,0.152687,0.104947,0.500000,0.047740,-141.155701,-182.760742,-8.368002,-7.267502
7,0.665509,-0.161607,-0.226753,0.750000,0.065146,-137.873260,-184.242737,-8.033894,-6.710013
8,0.618869,0.093261,-0.062893,1.000000,0.156154,-135.493912,-181.726562,-8.107296,-7.080783
9,0.510376,0.053770,-0.413740,0.750000,0.467511,-141.598328,-192.244492,-8.010671,-6.895947
10,0.494214,-0.027936,-0.501287,1.000000,0.473351,-132.787247,-183.588745,-8.278318,-7.185899


Unsloth: Restored added_tokens_decoder metadata in /tmp/dpo-repro/checkpoint-39/tokenizer_config.json.


TrainOutput(global_step=39, training_loss=0.2093821246212778, metrics={'train_runtime': 692.4582, 'train_samples_per_second': 0.225, 'train_steps_per_second': 0.056, 'total_flos': 0.0, 'train_loss': 0.2093821246212778, 'epoch': 3.0})

## Resolution — working stack (Apr 30, 2026)

After resolving 3 cascading upstream bugs (PR #5199 merged into mainline Apr 28) plus 1 usage-side requirement discovered during real DPO training, this stack runs vision DPO end-to-end on Gemma 4 E4B:

- `unsloth`: `git+https://github.com/unslothai/unsloth.git@4f9c8321a2136e62fd86fe722a544afd534334a5` (mainline post-merge)
- `trl`: 0.24.0 (NOT 0.22.2 from the original report — bug 3 lived there)
- `torch`: 2.11.0+cu129 + `torchvision`: 0.26.0+cu129 (matching CUDA builds, install with `--index-url https://download.pytorch.org/whl/cu129`)
- `transformers`: 5.5.0
- `datasets`: 4.3.0
- `peft`: 0.19.1
- `unsloth-zoo`: 2026.4.9
- `bitsandbytes`: 0.49.2

Install order matters — separate transactions so dependency resolution doesn't downgrade things. See cell 2 above for the working sequence.

**Plus the dataset-side requirement:** PROMPT must contain Gemma 4's image token (`tokenizer.image_token`, which is `<|image|>`). Without it, training fails at first step with `ValueError: Image features and image tokens do not match, tokens: 0, features: 512`. See cell 6 above for the required PROMPT pattern.

### Synthetic test result

39-step DPO run on Gemma 4 E4B 4-bit, vision-unfrozen LoRA r=16:

- Loss: 0.6931 (random baseline) → ~0.005 (converged)
- `rewards/accuracies`: 0.0 → 1.0 (sustained from step ~5 onwards)
- `rewards/margin`: 0 → ~6.5

The model overfits to the synthetic preferences as expected — confirms the training pipeline is working end-to-end.

### Note on commit pinning

This notebook was originally pinned to `datta0/unsloth@e96d05bad233052a30f894c5050eb7ec2e65ebc5` (a commit on Datta0's `dpo_mp_hang` branch). After PR #5199 merged on Apr 28, that commit was orphaned by force-push during PR cleanup — the commit ID still resolves but is no longer reachable from any branch on the fork.

The orphaned commit was MISSING fix #3's vision keys handling (added in `dc3e6a5`, which came after `6b11713` and before the merge). The synthetic test still passed on the orphaned pin because the synthetic data shape didn't exercise the missing code path. Real DPO training on production-shape data exposed the missing fix.

**Lesson:** pin to merge commits on official repos, not branch tips on contributor forks. Branch tips can be force-pushed; orphaned commits get GC'd.

### Original "What I've tried" (kept for posterity)

These were attempts during initial debugging on Apr 26 BEFORE the upstream fixes landed:

1. `dataset_num_proc=1` and `=2` — both hung. Single-process meant no fork issue, but still wedged.
2. Forced `HF_DATASETS_CACHE` to a known-writable persistent path — no change.
3. Explicit `Features({"images": [HFImage()], ...})` schema — Dataset constructed in 0s, trainer still hung at the same step.
4. Pre-tokenize via manual loop, then pass tokenized dataset — TRL's `_prepare_dataset` still called `dataset.map(..., remove_columns=["chosen", "rejected"])` and errored because those columns were gone.

None of these were the right path. The actual fix was upstream library code (the three commits in PR #5199) plus a TRL version bump.

### Credits

[@datta0](https://github.com/datta0) for the multiple fast turnarounds on the unsloth fixes. Issue thread: [#5196](https://github.com/unslothai/unsloth/issues/5196). PR: [#5199](https://github.com/unslothai/unsloth/pull/5199).